In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/react-2026-datathon/sample_submission.csv
/kaggle/input/competitions/react-2026-datathon/data_dictionary.csv
/kaggle/input/competitions/react-2026-datathon/train.csv
/kaggle/input/competitions/react-2026-datathon/test.csv


In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

In [3]:
DATA = '/kaggle/input/competitions/react-2026-datathon'

train = pd.read_csv(f'{DATA}/train.csv', parse_dates=['timestamp'])
test = pd.read_csv(f'{DATA}/test.csv', parse_dates=['timestamp'])

print(train.shape, test.shape, train.fraud.mean())

(731942, 13) (262648, 12) 0.0176052200857445


In [4]:
CATS = ['merchant_category', 'device_type', 'location', 'payment_method', 'transaction_type']
NULLABLE = ['merchant_category', 'device_type', 'location']
H, D = 3600, 86400

In [5]:
df = pd.concat([train, test.assign(fraud=np.nan)], ignore_index=True)

for c in NULLABLE:
    df[c + '_miss'] = df[c].isna().astype('int8')
    df[c] = df[c].fillna('__MISSING__')

df = df.sort_values(['timestamp', 'transaction_id']).reset_index(drop=True)
print(df.shape, df.timestamp.min(), df.timestamp.max())

(994590, 16) 2026-01-01 00:00:43 2026-09-15 22:34:38


In [6]:
def past_count(d, keys):
    return d.groupby(keys, observed=True).cumcount().astype('int32')

def past_nunique(d, key, val):
    first = (~d.duplicated([key, val])).astype('int32')
    cum = first.groupby(d[key], observed=True).cumsum()
    return (cum - first).astype('int32')

In [7]:
def prior_stats(codes, vals):
    s = pd.Series(vals)
    cnt = s.groupby(codes).cumcount().values
    csum = s.groupby(codes).cumsum().values - vals
    csq = (s ** 2).groupby(codes).cumsum().values - vals ** 2
    mean = np.where(cnt > 0, csum / np.maximum(cnt, 1), np.nan)
    var = np.where(cnt > 1, (csq - cnt * mean ** 2) / np.maximum(cnt - 1, 1), np.nan)
    return mean, np.sqrt(np.clip(var, 0, None)), cnt

In [8]:
def roll_counts(ts, codes, windows):
    order = np.lexsort((ts, codes))
    t_s, c_s = ts[order], codes[order]
    n = len(t_s)
    bounds = np.r_[0, np.flatnonzero(np.diff(c_s)) + 1, n]
    out = {w: np.empty(n, dtype=np.int32) for w in windows}
    for a, b in zip(bounds[:-1], bounds[1:]):
        t = t_s[a:b]
        pos = np.arange(b - a)
        for w in windows:
            out[w][a:b] = pos - np.searchsorted(t, t - w, side='left')
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    return {w: v[inv] for w, v in out.items()}

In [9]:
def sliding_nunique(ts, parent, child, window):
    order = np.lexsort((ts, parent))
    t_s, p_s, c_s = ts[order], parent[order], child[order]
    n = len(t_s)
    res = np.zeros(n, dtype=np.int32)
    bounds = np.r_[0, np.flatnonzero(np.diff(p_s)) + 1, n]
    for a, b in zip(bounds[:-1], bounds[1:]):
        t, c = t_s[a:b], c_s[a:b]
        cnt, left = {}, 0
        for i in range(b - a):
            while t[i] - t[left] >= window:
                v = c[left]
                cnt[v] -= 1
                if cnt[v] == 0:
                    del cnt[v]
                left += 1
            res[a + i] = len(cnt)
            cnt[c[i]] = cnt.get(c[i], 0) + 1
    inv = np.empty(n, dtype=np.int64)
    inv[order] = np.arange(n)
    return res[inv]

In [10]:
ts = df.timestamp
tsec = ts.values.astype('datetime64[s]').astype(np.int64)
amt = df.amount_bdt.values
lamt = np.log1p(amt)
g = df.groupby('customer_id', observed=True)

cust_code = pd.factorize(df.customer_id)[0]
dev_code = pd.factorize(df.device_id)[0]
mer_code = pd.factorize(df.merchant_id)[0]
loc_code = pd.factorize(df.location)[0]

In [11]:
f = pd.DataFrame(index=df.index)

f['amount'] = amt
f['log_amount'] = lamt
f['account_age_days'] = df.account_age_days
f['hour'] = ts.dt.hour
f['dow'] = ts.dt.dayofweek
f['is_night'] = ((f.hour >= 0) & (f.hour <= 4)).astype('int8')
f['amt_is_round100'] = (amt % 100 == 0).astype('int8')
f['amt_has_decimals'] = ((amt * 100) % 100 != 0).astype('int8')
f['amt_digits'] = np.floor(np.log10(np.maximum(amt, 1))).astype('int8')

In [12]:
f['cust_n_prior'] = past_count(df, 'customer_id')

for k in (1, 2, 3, 5):
    f[f'cust_dt{k}'] = (ts - g.timestamp.shift(k)).dt.total_seconds()

f['log_dt1'] = np.log1p(f.cust_dt1.clip(lower=0))

In [13]:
cm, cs, _ = prior_stats(cust_code, amt)
lcm, lcs, _ = prior_stats(cust_code, lamt)

f['cust_prior_mean'] = cm
f['cust_prior_std'] = cs
f['amt_ratio_mean'] = amt / cm
f['amt_z'] = (amt - cm) / np.where(cs > 1e-6, cs, np.nan)
f['lamt_z_cust'] = (lamt - lcm) / np.where(lcs > 1e-6, lcs, np.nan)
f['log_amt_ratio'] = lamt - np.log1p(cm)

In [14]:
prev_amt = g.amount_bdt.shift(1)
cummax = g.amount_bdt.transform(lambda s: s.shift(1).cummax())

f['prev_amt'] = prev_amt
f['amt_ratio_prev'] = amt / prev_amt
f['cust_prior_max'] = cummax
f['amt_ratio_max'] = amt / cummax

In [15]:
r = f.amt_ratio_mean
f['amt_ratio_gt2'] = (r > 2).astype('int8')
f['amt_ratio_gt5'] = (r > 5).astype('int8')
f['amt_ratio_gt10'] = (r > 10).astype('int8')
f['night_x_logamt'] = f.is_night * f.log_amount

In [16]:
f['cust_ndev_prior'] = past_nunique(df, 'customer_id', 'device_id')
f['cust_nloc_prior'] = past_nunique(df, 'customer_id', 'location')
f['cust_nmer_prior'] = past_nunique(df, 'customer_id', 'merchant_id')

for col, nm in [('device_id', 'device'), ('location', 'loc'), ('merchant_id', 'mer'),
                ('merchant_category', 'mcat'), ('payment_method', 'ptype')]:
    f[f'new_{nm}_for_cust'] = (past_count(df, ['customer_id', col]) == 0).astype('int8')

f['cust_devuse_n'] = past_count(df, ['customer_id', 'device_id'])
f['cust_locuse_n'] = past_count(df, ['customer_id', 'location'])

In [17]:
prev_loc = g.location.shift(1)
prev_dev = g.device_id.shift(1)

f['loc_changed'] = ((df.location != prev_loc) & prev_loc.notna()).astype('int8')
f['dev_changed'] = ((df.device_id != prev_dev) & prev_dev.notna()).astype('int8')
f['teleport'] = ((f.loc_changed == 1) & (f.cust_dt1 < 3600)).astype('int8')
f['loc_change_speed'] = f.loc_changed / (f.cust_dt1.fillna(86400) + 1)
f['newdev_x_ratio'] = f.new_device_for_cust * f.amt_ratio_mean.clip(-20, 20)

In [18]:
f['dev_n_prior'] = past_count(df, 'device_id')
f['dev_ncust_prior'] = past_nunique(df, 'device_id', 'customer_id')
f['dev_is_new'] = (f.dev_n_prior == 0).astype('int8')
f['dev_cust_ratio'] = f.dev_ncust_prior / (f.dev_n_prior + 1)
f['dev_dt'] = (ts - df.groupby('device_id', observed=True).timestamp.shift(1)).dt.total_seconds()

f['mer_n_prior'] = past_count(df, 'merchant_id')
f['mer_ncust_prior'] = past_nunique(df, 'merchant_id', 'customer_id')
f['loc_n_prior'] = past_count(df, 'location')

In [19]:
f['dev_ncust_30d'] = sliding_nunique(tsec, dev_code, cust_code, 30 * D)
f['dev_ncust_7d'] = sliding_nunique(tsec, dev_code, cust_code, 7 * D)
f['dev_ncust_30d_log'] = np.log1p(f.dev_ncust_30d)
f['dev_ncust_30d_high'] = (f.dev_ncust_30d >= 6).astype('int8')
f['cust_nloc_7d'] = sliding_nunique(tsec, cust_code, loc_code, 7 * D)

In [20]:
for key, short in [('customer_id', 'cust'), ('device_id', 'dev'), ('merchant_id', 'mer')]:
    codes = pd.factorize(df[key])[0]
    rc = roll_counts(tsec, codes, [H, 6 * H, D, 7 * D, 30 * D])
    for w, nm in [(H, '1h'), (6 * H, '6h'), (D, '1d'), (7 * D, '7d'), (30 * D, '30d')]:
        f[f'{short}_cnt_{nm}'] = rc[w]
    f[f'{short}_burst_1h_7d'] = rc[H] / (rc[7 * D] / 168 + 0.01)
    f[f'{short}_burst_1d_30d'] = rc[D] / (rc[30 * D] / 30 + 0.01)

In [21]:
for codes, short in [(mer_code, 'mer'), (dev_code, 'dev'), (loc_code, 'loc')]:
    mu, sd, _ = prior_stats(codes, lamt)
    f[f'lamt_z_{short}'] = (lamt - mu) / np.where(sd > 1e-6, sd, np.nan)
    f[f'lamt_diff_{short}'] = lamt - mu

In [22]:
pair = pd.factorize(df.customer_id.astype(str) + '|' + df.device_id.astype(str))[0]
rc = roll_counts(tsec, pair, [D, 7 * D])
f['custdev_cnt_1d'], f['custdev_cnt_7d'] = rc[D], rc[7 * D]

same_amt = pd.factorize(df.customer_id.astype(str) + '|' + df.amount_bdt.round(2).astype(str))[0]
f['cust_same_amt_prior'] = pd.Series(same_amt).groupby(same_amt).cumcount().values

In [23]:
for c in CATS:
    f[c] = df[c].astype('category')
for c in NULLABLE:
    f[c + '_miss'] = df[c + '_miss'].values

f = f.replace([np.inf, -np.inf], np.nan)
print(f.shape)

(994590, 95)


In [24]:
f_cat = f.copy()
for c in CATS:
    f_cat[c] = f_cat[c].astype(str)
cat_idx = [f_cat.columns.get_loc(c) for c in CATS]

f_num = f.copy()
for c in CATS:
    f_num[c] = f_num[c].cat.codes

In [25]:
y = df.fraud.values
is_train = df.fraud.notna().values

cut = pd.Timestamp('2026-07-01')
trm = is_train & (df.timestamp < cut).values
vam = is_train & (df.timestamp >= cut).values

print(trm.sum(), vam.sum(), int(y[vam].sum()))

672477 59465 926


In [26]:
lgb_p = dict(n_estimators=700, learning_rate=0.04, num_leaves=63, min_child_samples=100,
             subsample=0.85, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
             random_state=0, verbosity=-1)

m1 = lgb.LGBMClassifier(**lgb_p)
m1.fit(f[trm], y[trm], categorical_feature=CATS)
p1 = m1.predict_proba(f[vam])[:, 1]
print(average_precision_score(y[vam], p1))

0.5210648953351236


In [27]:
cb_p = dict(iterations=800, learning_rate=0.04, depth=7, l2_leaf_reg=6,
            random_seed=1, verbose=False, allow_writing_files=False)

m2 = CatBoostClassifier(**cb_p)
m2.fit(f_cat[trm], y[trm], cat_features=cat_idx)
p2 = m2.predict_proba(f_cat[vam])[:, 1]
print(average_precision_score(y[vam], p2))

0.5197285120587778


In [28]:
hgb_p = dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=40,
             l2_regularization=1.0, categorical_features=CATS, early_stopping=False,
             random_state=0)

m3 = HistGradientBoostingClassifier(**hgb_p)
m3.fit(f[trm], y[trm])
p3 = m3.predict_proba(f[vam])[:, 1]
print(average_precision_score(y[vam], p3))

0.5167647398273879


In [29]:
r1, r2, r3 = rankdata(p1), rankdata(p2), rankdata(p3)
print(pd.DataFrame({'lgb': r1, 'cat': r2, 'hgb': r3}).corr(method='spearman').round(3))
print(average_precision_score(y[vam], r1 + r2 + r3) / 1)

       lgb    cat    hgb
lgb  1.000  0.541  0.627
cat  0.541  1.000  0.473
hgb  0.627  0.473  1.000
0.5216418955929045


In [30]:
for w in [(1, 1, 1), (2, 1, 1), (1, 1, 2), (1, 0, 1), (1, 1, 0)]:
    s = average_precision_score(y[vam], w[0] * r1 + w[1] * r2 + w[2] * r3)
    print(w, round(s, 5))

(1, 1, 1) 0.52164
(2, 1, 1) 0.52166
(1, 1, 2) 0.52096
(1, 0, 1) 0.52018
(1, 1, 0) 0.52225


In [31]:
W = (1, 1, 1)
print(W, average_precision_score(y[vam], W[0] * r1 + W[1] * r2 + W[2] * r3))

(1, 1, 1) 0.5216418955929045


In [32]:
t1 = np.zeros((~is_train).sum())
for s in range(3):
    m = lgb.LGBMClassifier(**{**lgb_p, 'random_state': s})
    m.fit(f[is_train], y[is_train], categorical_feature=CATS)
    t1 += rankdata(m.predict_proba(f[~is_train])[:, 1])

In [33]:
t2 = np.zeros((~is_train).sum())
for s in range(3):
    m = CatBoostClassifier(**{**cb_p, 'random_seed': s})
    m.fit(f_cat[is_train], y[is_train], cat_features=cat_idx)
    t2 += rankdata(m.predict_proba(f_cat[~is_train])[:, 1])

In [34]:
t3 = np.zeros((~is_train).sum())
for s in range(3):
    m = HistGradientBoostingClassifier(**{**hgb_p, 'random_state': s})
    m.fit(f[is_train], y[is_train])
    t3 += rankdata(m.predict_proba(f[~is_train])[:, 1])

In [35]:
blend = rankdata(t1) + rankdata(t2) + rankdata(t3)
blend = (blend - blend.min()) / (blend.max() - blend.min())
print(blend.mean())

0.4998520860313348


In [36]:
sub = pd.DataFrame({'transaction_id': df.loc[~is_train, 'transaction_id'].values, 'fraud': blend})
sample = pd.read_csv(f'{DATA}/sample_submission.csv')
sub = sample[['transaction_id']].merge(sub, on='transaction_id', how='left')

assert sub.fraud.notna().all()
assert sub.fraud.between(0, 1).all()
assert len(sub) == len(sample)

sub.to_csv('submission.csv', index=False)
print(sub.shape, sub.fraud.mean())

(262648, 2) 0.4998520860313348
